# Week 18: Start with a Testable User Need

This notebook follows the reviewed Week 18 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. Start with a Testable User Need
2. Architecture Assigns Responsibilities to Components
3. Data Flow Reveals Trust Boundaries
4. Select Models from Evaluation Evidence
5. Use Structured Output, RAG, and Tools Only When Needed
6. Workflow State Makes Behaviour Inspectable
7. The API Contract Includes Failure Responses
8. Evaluation Covers Components and User Outcomes
9. Observability Connects a Result to Its Causes
10. Security and Cost Are Release Requirements
11. Release Includes Rollback and Failure Demonstrations
12. Capstone Deliverable: Evidence before Presentation

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. Start with a Testable User Need

A **requirement** is a testable statement of needed behaviour or constraint.

Define:

- user and decision;
- input and expected output;
- quality threshold;
- unsupported and denied behaviour;
- latency target;
- cost limit;
- privacy and security constraints;
- deployment environment;
- owner and success measure.

Choose a narrow problem whose evidence and failures can be demonstrated.

### Work it out first

Need: `Support staff must answer policy questions using the current approved handbook.`

Acceptance:

- citation-supported answer or insufficient evidence;
- no cross-tenant sources;
- p95 latency under `3s`;
- estimated cost under `$0.02` per request;
- zero unsupported critical claims in the reviewed test set.

### Notebook bridge

The selected capstone notebook supplies technical patterns, not the project requirements.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
acceptance = {
    "groundedness_min": 0.95,
    "p95_latency_ms_max": 3000,
    "cost_usd_max": 0.02,
    "cross_tenant_failures_max": 0,
}

Expected output:

```text
A measurable contract used throughout design and evaluation.
```


## 2. Architecture Assigns Responsibilities to Components

An **architecture** identifies components, responsibilities, interfaces, and constraints.

A capstone may include:

- web or API client;
- request validation;
- workflow state;
- retrieval and vector store;
- model provider;
- structured output;
- authorized tools;
- database;
- observability;
- deployment infrastructure.

Use the fewest components that satisfy requirements. Every additional service adds failure and cost.

### Work it out first

Policy assistant:

`Client -> FastAPI -> LangGraph workflow -> retriever -> model -> validated answer`

Documents enter through a separate ingestion job. The online API cannot modify source policies.

### Notebook bridge

Learners adapt only notebook components needed by the selected architecture.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
Online path: request -> validate -> retrieve -> answer -> verify -> respond
Offline path: source -> parse -> approve -> chunk -> embed -> index

Expected output:

```text
Separate online and ingestion responsibilities with explicit interfaces.
```


## 3. Data Flow Reveals Trust Boundaries

A **data-flow diagram** shows:

- data sources;
- movement between components;
- transformations;
- storage;
- external providers;
- trust boundaries;
- authentication and authorization points;
- sensitive fields;
- retention and deletion.

Untrusted user input, retrieved documents, model output, and tool results all require validation at their boundaries.

### Work it out first

Policy documents are approved internally, but their extracted text may contain parser errors. User question is untrusted. Model output is untrusted. Final response is released only after schema, citation, and authorization checks.

### Notebook bridge

Tool and RAG notebook code is placed only inside approved trust boundaries.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
[User] --untrusted question--> [API validation]
[Retriever] --authorized chunks--> [Model]
[Model] --untrusted draft--> [Output validation]

Expected output:

```text
Every boundary has an owner and required check.
```


## 4. Select Models from Evaluation Evidence

Model selection should compare:

- baseline;
- at least two suitable candidates;
- quality on the same test cases;
- structured-output and tool capability;
- latency and token use;
- cost;
- data policy;
- deployment fit.

Record exact model identifiers and configuration. A larger model is not automatically the correct production choice.

### Work it out first

Candidate A: groundedness `0.97`, p95 `4.5s`, cost `$0.03`  
Candidate B: groundedness `0.95`, p95 `2.2s`, cost `$0.01`

If minimum groundedness is `0.95`, p95 maximum `3s`, and cost maximum `$0.02`, only B passes all constraints.

### Notebook bridge

Provider comparison from Week 14 becomes a capstone selection report.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
passing = results[
    (results.groundedness >= 0.95)
    & (results.p95_ms <= 3000)
    & (results.cost <= 0.02)
]

Expected output:

```text
Only candidates satisfying every hard requirement.
```


## 5. Use Structured Output, RAG, and Tools Only When Needed

Use **structured output** when downstream code needs fields and types.

Use **RAG** when answers require current, private, or cited evidence.

Use **tools** when the system must retrieve live data or perform a bounded operation.

Use a **workflow** when steps are known. Use agent choice only when deciding among allowed actions adds measurable value.

Each capability needs its own validation and evaluation.

### Work it out first

Policy assistant:

- structured output: answer, status, citation IDs;
- RAG: approved handbook;
- tool: read-only ticket lookup only if ticket context is required;
- no write-capable agent because the requirement is answer support.

### Notebook bridge

Learners select one primary capstone notebook and use others only for justified components.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
class Answer(BaseModel):
    status: Literal["supported", "insufficient", "denied"]
    answer: str | None
    citation_ids: list[str]

Expected output:

```text
A typed outcome that represents success and safe non-success states.
```


## 6. Workflow State Makes Behaviour Inspectable

Capstone state should track:

- request and user context;
- validated inputs;
- retrieved evidence;
- model outputs;
- tool proposals and observations;
- attempts, tokens, latency, and cost;
- approvals;
- current status;
- final result and errors.

Declare terminal statuses such as `supported`, `insufficient`, `denied`, `rejected`, `timeout`, and `failed`.

### Work it out first

Route:

`validate -> retrieve`

- denied -> END
- no evidence -> insufficient -> END
- evidence -> generate -> verify
- unsupported draft -> one bounded repair
- supported -> END

### Notebook bridge

The LangGraph quickstart provides the graph-building mechanics.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
class CapstoneState(TypedDict):
    request_id: str
    evidence: list[Document]
    attempts: int
    cost: float
    status: str

Expected output:

```text
A state record sufficient to explain every route and final outcome.
```


## 7. The API Contract Includes Failure Responses

Define:

- request schema;
- response schema;
- authentication;
- authorization;
- status codes;
- idempotency where actions occur;
- request-size and rate limits;
- health checks;
- model and corpus versions.

Clients must distinguish supported answer, insufficient evidence, denied access, invalid input, and unavailable service.

### Work it out first

`200` supported answer  
`200` typed insufficient-evidence response when the request is valid but evidence is absent  
`403` denied authorization  
`422` invalid request  
`503` provider unavailable

The contract documents which body accompanies each status.

### Notebook bridge

Week 11 deployment patterns wrap the final workflow.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
@app.post("/answer", response_model=AnswerResponse)
def answer(request: AnswerRequest, user=Depends(authenticate)):
    return workflow.invoke(to_state(request, user))

Expected output:

```text
A typed API response derived from the workflow's final state.
```


## 8. Evaluation Covers Components and User Outcomes

Evaluate:

- request validation;
- retrieval precision and recall;
- groundedness and citation support;
- structured-output validity;
- tool selection and arguments;
- workflow path and stopping;
- latency and cost;
- security and access isolation;
- user task success.

Use a versioned dataset containing normal, difficult, unsupported, denied, and failure cases.

### Work it out first

Release thresholds:

- schema validity `100%`;
- citation IDs valid `100%`;
- groundedness at least `95%`;
- cross-tenant retrieval failures `0`;
- p95 latency under `3s`;
- p95 cost under `$0.02`.

### Notebook bridge

Week 16's evaluation harness becomes the capstone release gate.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
release_ok = all(check.passed for check in evaluation_report.checks)

Expected output:

```text
True only when every required component and end-to-end check passes.
```


## 9. Observability Connects a Result to Its Causes

For each request, record:

- request and trace ID;
- code, model, prompt, schema, and corpus versions;
- graph path and node timings;
- retrieved chunk IDs and ranks;
- tool calls and outcomes;
- validation results;
- input/output tokens and cost;
- final status and error category.

Monitor availability, latency, errors, cost, retrieval drift, and delayed model-quality signals.

### Work it out first

Latency trace:

validation `5ms`  
retrieval `120ms`  
generation `1,600ms`  
verification `80ms`

Total `1,805ms`; generation owns most latency.

### Notebook bridge

Tracing from Week 14 covers every graph node and external call.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
trace.set_attributes({
    "model_version": MODEL_VERSION,
    "corpus_version": CORPUS_VERSION,
    "final_status": state["status"],
})

Expected output:

```text
A trace that can reproduce the component path behind one result.
```


## 10. Security and Cost Are Release Requirements

Security controls:

- scoped identities and secrets;
- tenant-aware retrieval;
- input and output validation;
- tool allowlists and argument checks;
- human approval for high impact;
- dependency scanning;
- patching and rollback;
- log redaction.

Cost controls:

- token and tool-call budgets;
- rate limits;
- smallest suitable infrastructure;
- budget alerts;
- resource tags and cleanup;
- per-request and daily caps.

### Work it out first

Attack: document contains tool instructions requesting data export.

Controls:

- retrieved text remains untrusted;
- export tool absent from allowlist;
- model has no credentials;
- authorization denies cross-tenant data;
- trace records denial.

### Notebook bridge

The capstone integrates Week 11 deployment controls and Week 17 agent limits.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
if proposed_tool not in allowed_tools:
    return {"status": "denied", "reason": "tool_not_allowed"}

Expected output:

```text
A typed denial with no side effect and an auditable reason.
```


## 11. Release Includes Rollback and Failure Demonstrations

Before release, demonstrate:

- malformed request;
- unsupported question;
- unauthorized access;
- prompt injection;
- provider timeout;
- invalid model output;
- tool denial;
- budget exhaustion;
- health-check failure;
- rollback to previous artifact.

Create a runbook describing alerts, diagnosis, mitigation, rollback, owners, and cleanup.

### Work it out first

New corpus index reduces groundedness below threshold.

Rollback:

1. stop routing to new index version;
2. restore previous version alias;
3. verify health and test cases;
4. record incident and failed changes;
5. correct ingestion before another release.

### Notebook bridge

The final notebook workflow is packaged with versioned model, corpus, prompt, and graph artifacts.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
active_index -> policy-v3
candidate_index -> policy-v4
rollback changes active alias back to policy-v3

Expected output:

```text
The previous verified artifact serves requests without rebuilding during the incident.
```


## 12. Capstone Deliverable: Evidence before Presentation

Submit:

1. problem statement and acceptance criteria;
2. architecture and data-flow diagrams;
3. threat model and cost estimate;
4. versioned source, configuration, and environment;
5. model and retrieval selection report;
6. API and workflow contracts;
7. evaluation dataset and results;
8. failure-analysis report;
9. deployment and health evidence;
10. traces, monitoring, and budget alerts;
11. runbook and rollback proof;
12. limitations and responsible-use statement.

Demonstrate failure paths before the polished success path.

### Work it out first

Final demonstration:

1. denied cross-tenant request;
2. unsupported question;
3. malformed output caught by validation;
4. provider timeout and controlled response;
5. grounded answer with verified citations;
6. rollback to previous artifact.

### Notebook bridge

Use one selected agentic capstone notebook as a starting pattern and document every modernization or replacement.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
assert release_report.all_required_checks_pass
assert release_report.rollback_verified

Expected output:

```text
Both assertions pass before the capstone is marked production-ready.
```


## Guided lab

Submit:

1. problem statement and acceptance criteria;
2. architecture and data-flow diagrams;
3. threat model and cost estimate;
4. versioned source, configuration, and environment;
5. model and retrieval selection report;
6. API and workflow contracts;
7. evaluation dataset and results;
8. failure-analysis report;
9. deployment and health evidence;
10. traces, monitoring, and budget alerts;
11. runbook and rollback proof;
12. limitations and responsible-use statement.

Demonstrate failure paths before the polished success path.

### Reference result

Final demonstration:

1. denied cross-tenant request;
2. unsupported question;
3. malformed output caught by validation;
4. provider timeout and controlled response;
5. grounded answer with verified citations;
6. rollback to previous artifact.


In [ ]:
# Guided lab workspace: Week 18
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://www.nist.gov/itl/ai-risk-management-framework>
- <https://docs.langchain.com/langsmith/evaluation-concepts>
- <https://docs.aws.amazon.com/wellarchitected/latest/framework/welcome.html>
- <https://docs.langchain.com/oss/python/langgraph/overview>
- <https://cheatsheetseries.owasp.org/cheatsheets/Threat_Modeling_Cheat_Sheet.html>
- <https://docs.langchain.com/langsmith/evaluation>
- <https://docs.litellm.ai/>
- <https://docs.langchain.com/oss/python/langchain/overview>
- <https://docs.langchain.com/oss/python/langgraph/graph-api>
- <https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph>
- <https://fastapi.tiangolo.com/tutorial/>
- <https://fastapi.tiangolo.com/tutorial/response-status-code/>
- <https://docs.langchain.com/langsmith/evaluate-rag-tutorial>
- <https://docs.langchain.com/langsmith/observability>
- <https://opentelemetry.io/docs/concepts/>
- <https://genai.owasp.org/llm-top-10/>
- <https://docs.aws.amazon.com/hands-on/latest/control-your-costs-free-tier-budgets/control-your-costs-free-tier-budgets.html>
- <https://docs.aws.amazon.com/wellarchitected/latest/reliability-pillar/welcome.html>
- <https://github.com/curiousily/AI-Bootcamp>